# 05 — Sentiment Analysis: Load & Clean

Collects newspaper headlines for **Jordan**, **Mozambique**, and **Zambia** via the MediaCloud API, preprocesses them, and scores sentiment using Llama 3.

**What this notebook does:**
1. Identify the three countries with the largest USAID cuts that have active press indexed by MediaCloud
2. Fetch headlines mentioning China or the US (pre- and post-inauguration)
3. Preprocess: parse dates, assign pre/post period, filter by keyword
4. Score sentiment using a locally-run **Llama 3** model via Ollama
5. Save `df_china_scored.csv` and `df_us_scored.csv` — upload these to Google Drive and add the file IDs to `06_sentiment_analysis.ipynb`

## Helper Functions

| Function | Signature | What it does |
|---|---|---|
| `contains_any` | `contains_any(text, keywords)` | Returns `True` if any keyword from `keywords` appears (case-insensitive) in `text`. Used to filter headlines by language-specific China / US keyword lists (English, Arabic, Portuguese). |
| `fetch_stories` | `fetch_stories(search_api, query, collection_id, start, end, max_stories=500)` | Paginates through the MediaCloud search API for a given query, collection, and date range and returns a flat list of story dicts. |
| `classify_sentiment` | `classify_sentiment(title)` | Sends a headline to a locally-running Llama 3 model via Ollama and returns `"Positive"`, `"Neutral"`, or `"Negative"`. Falls back to `"Neutral"` on timeout. |
| `score_headlines` | `score_headlines(df)` | Applies `classify_sentiment` to every row of a headlines DataFrame with a `tqdm` progress bar. Returns the DataFrame with a new `sentiment` column. |

## 0. Install & import

In [ ]:
import os
import datetime
import warnings
warnings.filterwarnings('ignore')

import requests            # used by classify_sentiment (Ollama HTTP API)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gdown
from tqdm.auto import tqdm  # used by score_headlines (progress bar)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

---
## 1. Which countries saw the largest aid cuts? (context)

This section identifies the countries we focus on. It replicates the aid-drop calculation from `01_load_clean.ipynb` so this notebook is self-contained.

In [5]:
gdown.download(id="1Sw8kegqF-n9TIbTjc9Nq-n0wO4WtoGor", output="df_clean.csv", quiet=False)
df = pd.read_csv("df_clean.csv", low_memory=False)

# Aid/GDP ratio per country, 2024 vs 2025
aid_ratio = df[['ms_name', 'year', 'aid_gdp_ratio']].drop_duplicates()
aid_2024  = aid_ratio[aid_ratio['year'] == 2024].set_index('ms_name')['aid_gdp_ratio']
aid_2025  = aid_ratio[aid_ratio['year'] == 2025].set_index('ms_name')['aid_gdp_ratio']

cuts = (
    pd.DataFrame({'ratio_2024': aid_2024, 'ratio_2025': aid_2025})
    .dropna()
    .assign(ppt_change=lambda x: x['ratio_2025'] - x['ratio_2024'])
    .sort_values('ppt_change')
)

print("Largest drops in Aid disbursements as % of GDP (2024 → 2025):")
cuts.head(40)



Downloading...
From: https://drive.google.com/uc?id=1Sw8kegqF-n9TIbTjc9Nq-n0wO4WtoGor
To: /Users/charlesphillips/Documents/GitHub/qss20-final-project/code/df_clean.csv
100%|██████████████████████████████████████| 38.8M/38.8M [00:01<00:00, 26.4MB/s]


'df_clean.csv'

Largest drops in Aid disbursements as % of GDP (2024 → 2025):


,ratio_2024,ratio_2025,ppt_change
ms_name,,,
MARSHALL ISLANDS,150.204147,97.852206,-52.351942
PALAU,36.239114,26.055509,-10.183605
MICRONESIA (FEDERATED STATES OF),77.143602,72.931610,-4.211993
SOMALIA,8.043437,3.954794,-4.088642
SOUTH SUDAN,7.013478,3.962021,-3.051457
LESOTHO,4.938364,2.416217,-2.522147
UKRAINE,5.326919,3.398709,-1.928211
CENTRAL AFRICAN REPUBLIC,4.281262,2.361252,-1.920010
YEMEN,2.941885,1.219184,-1.722701


**Countries chosen for sentiment analysis:** Jordan, Mozambique, and Zambia.
All three experienced substantial aid cuts and have active press indexed by MediaCloud. Jordan's press is primarily Arabic, Mozambique's is Portuguese, and Zambia's is English.

---
## 2. Collect headlines via MediaCloud API

### 2a. Setup

You need a free API key from [mediacloud.org](https://search.mediacloud.org/). Set it as an environment variable:
```bash
export MC_API_KEY="your_key_here"
```
or paste it directly into the cell below.

In [6]:
# ── Paste your MediaCloud API key here ───────────────────────────────────
MC_API_KEY = '1c2399fcd6a37e40f249c8dfb7d0f3a5a7c03237'
# ─────────────────────────────────────────────────────────────────────────

# Date split: Trump's inauguration
INAUGURATION = datetime.date(2025, 1, 20)
PRE_START    = datetime.date(2024, 1, 1)   # a year and a month before
POST_END     = datetime.date(2026, 5, 28)   # today

# How many stories to fetch per country per query (increase for richer analysis)
MAX_STORIES  = 500

USE_API = bool(MC_API_KEY and MC_API_KEY.strip())
print(f"API key found: {USE_API}")
if not USE_API:
    print("Paste your key into MC_API_KEY above, then re-run this cell.")

API key found: True


### 2b. Discover collection IDs for Jordan, Mozambique, and Zambia

MediaCloud organises sources into *collections* (e.g. 'Jordan - National'). Run this cell to find the right IDs, then hard-code them in **Section 2c**.

In [7]:
if USE_API:
    from mediacloud.api import DirectoryApi
    dir_api = DirectoryApi(MC_API_KEY)

    for country_name in ['Jordan', 'Mozambique', 'Zambia']:
        print(f'\n── {country_name} collections ──')
        results = dir_api.collection_list(name=country_name, limit=10)
        for c in results.get('results', results.get('list', [])):
            print(f"  id={c['id']}  name={c['name']}")
else:
    print('Skipping — no API key.')


── Jordan collections ──
  id=34412072  name=Jordan - National
  id=38380537  name=Sogn og Fjordane, Norway - State & Local
  id=38380252  name=Jordan - State & Local
  id=38380253  name=Irbid, Jordan - State & Local

── Mozambique collections ──
  id=34412248  name=Mozambique - National
  id=38380426  name=Maputo, Mozambique - State & Local
  id=38380425  name=Mozambique - State & Local
  id=38380428  name=Zambezia, Mozambique - State & Local

── Zambia collections ──
  id=34412396  name=Zambia - National
  id=38381675  name=Zambia - State & Local
  id=38381676  name=Lusaka, Zambia - State & Local
  id=38381680  name=Eastern, Zambia - State & Local
  id=38381678  name=Copperbelt, Zambia - State & Local


### 2c. Fetch headlines mentioning China and the US

In [8]:
# ── Fill in collection IDs from 2b above ─────────────────────────────────
COLLECTION_IDS = {
    'Jordan':     34412072,   # paste ID found above
    'Mozambique': 34412248,   # paste ID found above
    'Zambia':     34412396,   # paste ID found above
}

# Search queries — English + Arabic (Jordan) + Portuguese (Mozambique)
QUERY_CHINA = (
    '"China" OR "Chinese" OR "Beijing" OR '
    '"\u0627\u0644\u0635\u064a\u0646" OR "\u0628\u0643\u064a\u0646" OR ' #Arabic
    '"Pequim" OR "chin\u00eas" OR "chinesa"'
)
QUERY_US = (
    '"United States" OR "America" OR "American" OR "Washington" OR "U.S." OR '
    '"\u0627\u0644\u0648\u0644\u0627\u064a\u0627\u062a \u0627\u0644\u0645\u062a\u062d\u062f\u0629" OR "\u0623\u0645\u0631\u064a\u0643\u0627" OR '
    '"Estados Unidos" OR "americano" OR "americana"'
)
# ─────────────────────────────────────────────────────────────────────────

def fetch_stories(search_api, query, collection_id, start, end, max_stories=500):
    """Paginate through MediaCloud results and return a flat list of story dicts."""
    all_stories = []
    page_token  = None
    while len(all_stories) < max_stories:
        batch, page_token = search_api.story_list(
            query,
            start_date=start,
            end_date=end,
            collection_ids=[collection_id],
            page_size=min(100, max_stories - len(all_stories)),
            pagination_token=page_token
        )
        all_stories.extend(batch)
        if page_token is None or len(batch) == 0:
            break
    return all_stories


if USE_API and all(v is not None for v in COLLECTION_IDS.values()):
    from mediacloud.api import SearchApi
    search_api = SearchApi(MC_API_KEY)

    api_frames = []
    for country, coll_id in COLLECTION_IDS.items():
        for topic, query in [('china', QUERY_CHINA), ('us', QUERY_US)]:
            for period, (start, end) in [
                ('pre',  (PRE_START, INAUGURATION - __import__('datetime').timedelta(days=1))),
                ('post', (INAUGURATION, POST_END))
            ]:
                stories = fetch_stories(search_api, query, coll_id, start, end, MAX_STORIES)
                print(f'  {country} | {topic} | {period}: {len(stories)} stories')
                for s in stories:
                    api_frames.append({
                        'country':      country,
                        'topic':        topic,
                        'period':       period,
                        'publish_date': s.get('publish_date', s.get('publication_date', '')),
                        'title':        s.get('title', ''),
                        'language':     s.get('language', ''),
                        'media_name':   s.get('media_name', ''),
                        'url':          s.get('url', ''),
                    })

    df_api = __import__('pandas').DataFrame(api_frames)
    print(f'\nTotal stories fetched: {len(df_api)}')
    df_api.to_csv('mediacloud_headlines.csv', index=False)
    print('Saved -> mediacloud_headlines.csv')
else:
    print('Fill in COLLECTION_IDS above, then re-run.')

  Jordan | china | pre: 500 stories
  Jordan | china | post: 500 stories
  Jordan | us | pre: 500 stories
  Jordan | us | post: 500 stories
  Mozambique | china | pre: 500 stories
  Mozambique | china | post: 500 stories
  Mozambique | us | pre: 500 stories
  Mozambique | us | post: 500 stories
  Zambia | china | pre: 500 stories
  Zambia | china | post: 500 stories
  Zambia | us | pre: 500 stories
  Zambia | us | post: 500 stories

Total stories fetched: 6000
Saved -> mediacloud_headlines.csv


In [10]:
# Load headlines into df_headlines
# If the API was just run above, use df_api directly; otherwise load from the saved CSV
if 'df_api' in vars():
    df_headlines = df_api.copy()
else:
    df_headlines = pd.read_csv('mediacloud_headlines.csv')

print(f"Loaded {len(df_headlines)} headlines")
print(df_headlines[['country', 'topic', 'period']].value_counts().sort_index())

Loaded 6000 headlines
country     topic  period
Jordan      china  post      500
                   pre       500
            us     post      500
                   pre       500
Mozambique  china  post      500
                   pre       500
            us     post      500
                   pre       500
Zambia      china  post      500
                   pre       500
            us     post      500
                   pre       500
Name: count, dtype: int64


---
## 3. Preprocess: dates, periods, and keyword filtering

In [11]:
# Parse dates
df_headlines['publish_date'] = pd.to_datetime(
    df_headlines['publish_date'], errors='coerce', utc=True
).dt.tz_localize(None)  # drop tz for easy comparison

df_headlines = df_headlines.dropna(subset=['publish_date', 'title'])
df_headlines['title'] = df_headlines['title'].astype(str).str.strip()
df_headlines = df_headlines[df_headlines['title'].str.len() > 5]  # drop empty/junk

# Pre / post inauguration flag
INAUGURATION_DT = pd.Timestamp('2025-01-20')
df_headlines['period'] = np.where(
    df_headlines['publish_date'] >= INAUGURATION_DT, 'Post-inauguration', 'Pre-inauguration'
)

print(df_headlines.groupby(['country', 'period']).size().unstack(fill_value=0))

period      Post-inauguration  Pre-inauguration
country                                        
Jordan                    999               999
Mozambique               1000              1000
Zambia                   1000              1000


In [12]:
# ── Keyword lists (English + Arabic for Jordan + Portuguese for Mozambique) ─
CHINA_KEYWORDS = [
    'china', 'chinese', 'beijing',                       # English / Zambia
    'الصين', 'الصيني', 'الصينية', 'بكين',             # Arabic / Jordan
    'pequim', 'chinês', 'chinesa',                        # Portuguese / Mozambique
]
US_KEYWORDS = [
    'united states', 'america', 'american', 'washington', 'u.s.',   # English
    'الولايات المتحدة', 'أمريكا', 'الأمريكي', 'الأمريكية',  # Arabic
    'estados unidos', 'americano', 'americana',                     # Portuguese
]
# ─────────────────────────────────────────────────────────────────────────

def contains_any(text, keywords):
    t = str(text).lower()
    return any(kw in t for kw in keywords)

if 'topic' not in df_headlines.columns:
    df_headlines['mentions_china'] = df_headlines['title'].apply(lambda t: contains_any(t, CHINA_KEYWORDS))
    df_headlines['mentions_us']    = df_headlines['title'].apply(lambda t: contains_any(t, US_KEYWORDS))
else:
    df_headlines['mentions_china'] = df_headlines['topic'] == 'china'
    df_headlines['mentions_us']    = df_headlines['topic'] == 'us'

df_china = df_headlines[df_headlines['mentions_china']].copy()
df_us    = df_headlines[df_headlines['mentions_us']].copy()

print(f'Headlines mentioning China: {len(df_china)}')
print(df_china.groupby(['country', 'period']).size().unstack(fill_value=0))

print(f'\nHeadlines mentioning the US: {len(df_us)}')
print(df_us.groupby(['country', 'period']).size().unstack(fill_value=0))

Headlines mentioning China: 3000
period      Post-inauguration  Pre-inauguration
country                                        
Jordan                    500               500
Mozambique                500               500
Zambia                    500               500

Headlines mentioning the US: 2998
period      Post-inauguration  Pre-inauguration
country                                        
Jordan                    499               499
Mozambique                500               500
Zambia                    500               500


---
## 4. Sentiment Analysis

We use a locally-run **Llama 3** model via Ollama to classify each headline as **Positive**, **Neutral**, or **Negative**. Llama 3 handles Arabic (Jordan), Portuguese (Mozambique), and English (Zambia) natively.

Ollama must be running locally (`ollama serve`) with the Llama 3 model pulled (`ollama pull llama3`) before executing the cells below.

In [13]:
import requests

# ── Customisable ────────────────────────────────────────────────────────
OLLAMA_URL   = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'llama3'   # change to 'llama3.1', 'mistral', etc. if needed
# ────────────────────────────────────────────────────────────────────────

SENTIMENT_PROMPT = (
    'Classify the sentiment of this news headline as exactly one of: '
    'Positive, Neutral, Negative.\n'
    'Respond with ONLY the label, nothing else.\n\n'
    'Headline: {title}'
)

def classify_sentiment(title):
    try:
        resp = requests.post(
            OLLAMA_URL,
            json={'model': OLLAMA_MODEL,
                  'prompt': SENTIMENT_PROMPT.format(title=title),
                  'stream': False},
            timeout=30
        )
        label = resp.json()['response'].strip().lower()
        if 'positive' in label: return 'Positive'
        if 'negative' in label: return 'Negative'
        return 'Neutral'
    except Exception:
        return 'Neutral'  # fallback on timeout / server down

# Sanity check — confirm Ollama is running before scoring everything
test = [
        ('China pledges $1 billion in aid to Zambia',  'English'),
    ('US sanctions devastate local economy',        'English'),
    ('الصين تستثمر في قطاع الطاقة بالأردن',       'Arabic'),
    ('China aumenta investimentos em Moçambique',   'Portuguese'),
]
print('Sanity check (if these hang, Ollama is not running):')
for title, lang in test:
    label = classify_sentiment(title)
    print(f'  [{label:8s}] ({lang}) {title}')

Sanity check (if these hang, Ollama is not running):
  [Positive] (English) China pledges $1 billion in aid to Zambia
  [Negative] (English) US sanctions devastate local economy
  [Neutral ] (Arabic) الصين تستثمر في قطاع الطاقة بالأردن
  [Positive] (Portuguese) China aumenta investimentos em Moçambique


In [14]:
from tqdm.auto import tqdm

def score_headlines(df):
    df = df.copy()
    labels = [classify_sentiment(t) for t in tqdm(df['title'], desc='Scoring')]
    df['sentiment'] = labels
    df['score']     = 1.0  # Llama doesn't return a confidence score
    return df

print('Scoring China headlines...')
df_china = score_headlines(df_china)

print('Scoring US headlines...')
df_us = score_headlines(df_us)

print('\nChina headline sentiment distribution:')
print(df_china['sentiment'].value_counts())
print('\nUS headline sentiment distribution:')
print(df_us['sentiment'].value_counts())

Scoring China headlines...


Scoring:   0%|          | 0/3000 [00:00<?, ?it/s]

Scoring US headlines...


Scoring:   0%|          | 0/2998 [00:00<?, ?it/s]


China headline sentiment distribution:
sentiment
Negative    1374
Positive     835
Neutral      791
Name: count, dtype: int64

US headline sentiment distribution:
sentiment
Negative    1500
Neutral      785
Positive     713
Name: count, dtype: int64


In [15]:
# Sample of scored headlines — a few from each country and topic
sample = (
    pd.concat([df_china.assign(topic='China'), df_us.assign(topic='US')])
    .groupby(['country', 'topic', 'sentiment'], group_keys=False)
    .apply(lambda g: g.sample(min(2, len(g)), random_state=42))
    .reset_index(drop=True)
    [['country', 'topic', 'sentiment', 'title']]
    .sort_values(['country', 'topic', 'sentiment'])
)
sample

,country,topic,sentiment,title
0,Jordan,China,Negative,الصين.. مقتل 8 أشخاص بحادث منجم للفحم (فيديو)
1,Jordan,China,Negative,روسيا تطالب بفتح مضيق هرمز أمام حركة النفط والغاز
2,Jordan,China,Neutral,أسرار العلاج بالحجامة.. كيف تعمل وما فوائدها؟
3,Jordan,China,Neutral,الخارجية الأميركية: روبيو بحث مع غوتيريش ملف م...
4,Jordan,China,Positive,"الإمارات تعلن ""تسريع"" بناء خط أنابيب نفط للالت..."
5,Jordan,China,Positive,القطاع الخاص القطري يعزز صادراته إلى الأردن بـ...
6,Jordan,US,Negative,ماذا قدم الأردن لغزة !؟
7,Jordan,US,Negative,Los Angeles wildfires: Which celebrities have ...
8,Jordan,US,Neutral,Queen Rania meets incoming First Lady Melania ...
9,Jordan,US,Neutral,المحكمة العليا الأميركية تنظر في قانون حظر تيك...


In [16]:
## Save the dfs
df_china.to_csv('df_china_scored.csv', index=False)
df_us.to_csv('df_us_scored.csv', index=False)